# SupportIQ — Free GPU QLoRA Fine-Tuning Engine (Kaggle / Colab)

> **Welcome to hands-on LLM fine-tuning!**
> In this notebook, you will personally fine-tune a Qwen LLM on the SupportIQ customer support dataset using **QLoRA (Quantized Low-Rank Adaptation)** on a **100% Free GPU**.
>
> **What You Will Accomplish:**
> 1. Load a Qwen base model in **4-bit precision (NF4)** using BitsAndBytes.
> 2. Attach trainable **LoRA adapters** (freezing 98.5% of base weights).
> 3. Train on conversational dialogues using Hugging Face **TRL `SFTTrainer`**.
> 4. Watch the training loss drop in real time.
> 5. Test the fine-tuned model live on your own custom queries.
> 6. Export and download the final **~50 MB adapter weights**.


### Step 1: Verify GPU Environment
Check that an NVIDIA GPU (e.g. Tesla T4 16GB) is attached and available.
*(On Kaggle: Settings -> Accelerator -> GPU T4 x2. On Colab: Runtime -> Change runtime type -> T4 GPU).*


In [ ]:
!nvidia-smi
import torch

print(f"\nPyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


### Step 2: Install Fine-Tuning Dependencies
Install Hugging Face `transformers`, `peft` (LoRA), `trl` (SFTTrainer), and `bitsandbytes` (4-bit quantization).


In [ ]:
!pip install -q --upgrade \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    trl>=0.8.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.28.0 \
    datasets>=2.18.0

print("Dependencies installed successfully!")


### Step 3: Prepare Dataset Files
Ensure `train.jsonl` and `val.jsonl` are present.
*(You can upload them directly using Kaggle's 'Add Data' or Colab's file upload sidebar).*


In [ ]:
import json
import os

# Check if data files are in current directory, or provide upload fallback
train_file = "train.jsonl"
val_file = "val.jsonl"

if not os.path.exists(train_file):
    print("Notice: train.jsonl not found in current directory.")
    print("Please upload train.jsonl and val.jsonl using the file upload sidebar!")
else:
    with open(train_file) as f:
        num_train = sum(1 for _ in f)
    with open(val_file) as f:
        num_val = sum(1 for _ in f)
    print(f"Found {num_train:,} training records and {num_val:,} validation records.")

    # Preview one formatted conversation
    with open(train_file) as f:
        sample = json.loads(f.readline())
    print("\nSample SFT Record Structure:")
    print(json.dumps(sample, indent=2)[:400] + "...\n}")


### Step 4: Load Qwen in 4-Bit Quantization (NF4)
Using `BitsAndBytesConfig`, we compress the 16-bit model weights into 4 bits, drastically cutting VRAM from ~8 GB down to **under 2.5 GB**!


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model choice: Qwen/Qwen2.5-0.5B (super fast dev) or Qwen/Qwen2.5-1.5B / Qwen/Qwen2.5-3B
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(f"Loading {MODEL_NAME} in 4-bit precision...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True,
)

print(f"Base model loaded successfully onto {model.device}!")


### Step 5: Attach Parameter-Efficient LoRA Adapter
We freeze 98.5%+ of the model's billions of parameters and only train lightweight rank-16 adapter matrices attached to the attention and MLP projection layers.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for 4-bit training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
print("=== TRAINABLE PARAMETERS ===")
model.print_trainable_parameters()


### Step 6: Train the Model with SFTTrainer
Run training. Watch the loss decrease with each step!
*(For a quick smoke test on Kaggle, 100-200 steps takes ~3 minutes. For a full run, set epochs=1).*


In [ ]:
from datasets import load_dataset

# TRL >=0.9: Use SFTConfig (replaces TrainingArguments + SFT kwargs)
# dataset_text_field and max_seq_length now live inside SFTConfig, NOT SFTTrainer
from trl import SFTConfig, SFTTrainer

# Load JSONL dataset
dataset = load_dataset(
    "json",
    data_files={"train": train_file, "validation": val_file},
)

OUTPUT_DIR = "supportiq_qlora_checkpoints"

# SFTConfig = TrainingArguments + SFT-specific args merged into one class
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=150,  # Change to num_train_epochs=1 for a full run!
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    optim="paged_adamw_8bit",
    report_to="none",
    # SFT-specific args now live here:
    max_seq_length=512,
    # Our dataset has a 'messages' key with [{role, content}] list.
    # TRL auto-detects this and applies the Qwen chat template — no
    # dataset_text_field needed for conversational format.
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=lora_config,
    args=sft_config,
    # tokenizer arg removed in TRL >=0.10 — use processing_class instead
    processing_class=tokenizer,
)

print("Starting QLoRA Fine-Tuning...")
trainer.train()
print("\nTraining finished successfully!")


### Step 7: Test Your Fine-Tuned Model Live!
Test your model with customer inquiries and watch it output structured JSON triage metadata and polite responses!


In [ ]:
system_prompt = (
    "You are SupportIQ, an expert customer support triage assistant. "
    "Classify the customer request into category and intent, and provide a helpful, polite response."
)

test_queries = [
    "How do I cancel my order #9981?",
    "I forgot my password, can you help me reset it?",
    "Where is my package? The tracking number says delivered but it's not here.",
]

print("=== BEFORE vs AFTER FINE-TUNING COMPARISON ===\n")
model.eval()

for query in test_queries:
    prompt = (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{query}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 1. BEFORE FINE-TUNING: Disable LoRA adapter (Raw Base Model)
    with torch.no_grad():
        with model.disable_adapter():
            out_base = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen_base = tokenizer.decode(out_base[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # 2. AFTER FINE-TUNING: Enable LoRA adapter (SupportIQ Fine-Tuned)
    with torch.no_grad():
        out_lora = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
        gen_lora = tokenizer.decode(out_lora[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    print(f"📌 CUSTOMER INQUIRY: {query}\n")
    print(f"❌ BEFORE FINE-TUNING (Raw Base Model):\n{gen_base.strip()}\n")
    print(f"✅ AFTER FINE-TUNING (SupportIQ LoRA):\n{gen_lora.strip()}\n")
    print("=" * 60 + "\n")


### Step 8: Save & Download Your Trained Adapter Weights
Save your final LoRA adapter (~50 MB) and zip it so you can download it to your local machine!


In [ ]:
import shutil

FINAL_ADAPTER_DIR = "supportiq_final_adapter"
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

shutil.make_archive("supportiq_final_adapter", "zip", FINAL_ADAPTER_DIR)
print("SUCCESS! Adapter saved and zipped to: supportiq_final_adapter.zip")
print(f"File size: {os.path.getsize('supportiq_final_adapter.zip') / (1024*1024):.2f} MB")
print("You can now download 'supportiq_final_adapter.zip' from the file browser!")
